# Stage 3 (CPU) -- the four-backbone grid

**No GPU needed**, and it runs one cell at a time.

Two corrections to what I first told you. The ~3 h estimate counted only the conformal evaluation
and omitted head fitting entirely; measured, the head fits add ~1.6 h, of which `groupdro_ll` on
CelebA/ResNet alone is 63 min. **The real figure is ~5.5 h.**

And the first version held all eight (backbone, dataset) cells in memory at once -- 3.3 GB of
features -- while `fit_groupdro_ll` casts the CelebA/ResNet training matrix to float64 and
standardises it, adding ~5 GB. Together that exhausted the runtime. The grid now streams one cell
at a time and **writes each finished cell to Drive**, so a disconnect costs at most the cell in
flight; re-run the grid cell to continue.

This one run unblocks four reviewer points:

| point | what this run supplies |
|---|---|
| **R1.3** | four backbones spanning two architecture families and three pretraining regimes |
| **R1.4** | five training seeds instead of the submitted three |
| **R2.4** | gated-out arms are evaluated and kept, so their influence can be measured |
| **R2.3 / R3.2** | regenerates `grid_records.csv`, which was lost with an ephemeral runtime and is what the hierarchical-bootstrap and correlation-CI re-analysis needs |

It also recovers the 50k CelebA ablation for free, before those features are deleted.

**Use a CPU runtime.** If a cache is missing, a cell aborts rather than quietly training a
ResNet-50 on CPU for hours.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # the Drive-cached zip is used first; no credential needed
CELEBA_DRIVE  = ""

# --- the grid ----------------------------------------------------------------------
# Four backbones spanning two architecture families AND three pretraining regimes, which is what
# makes R1.3 ("only two backbones limits the generalizability") an answered point rather than a
# conceded one:
#   resnet50_erm   CNN  supervised, fine-tuned in-domain
#   clip_vitb32    ViT  image-text contrastive
#   dinov2_vitb14  ViT  self-supervised, never saw a label
#   vit_b16_in1k   ViT  supervised ImageNet
BACKBONES = ("resnet50_erm", "clip_vitb32", "dinov2_vitb14", "vit_b16_in1k")
DATASETS  = ("waterbirds", "celeba")
SEEDS     = (0, 1, 2, 3, 4)    # R1.4 asked for more than the submitted three
METHODS   = ("erm", "dfr", "afr", "groupdro_ll", "balanced_subsample")
SCORES    = ("APS", "RAPS", "THR")
RHO_SWEEP = (0.95, 0.9, 0.8, 0.7, 0.6, 0.5)
N_SPLITS  = 10
ALPHA     = 0.1

CELEBA_RESNET_MAX_TRAIN = 30000   # must MATCH the paper's original run, or the ResNet cache misses
                                  # and this CPU runtime would start training a ResNet-50.

# The 50k CelebA representation ablation (optional, free: pure cache hits). It recovers the records
# for the abandoned subsample protocol, under which three cells read "marginal WINS" -- a
# counterexample that disappears at full train. Worth keeping as evidence that the full-train
# decision was the right one; the 7.5 GB of features can then be deleted.
RUN_50K_ABLATION = True
CEL50_EPOCHS = {"erm": 5, "reweight": 5, "groupdro": 8}   # the old budget, to match the old keys
CEL50_SEEDS  = (0, 1, 2)

## 1. Drive, repo, caches

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed: {cmd}")
    return r.returncode == 0

def init_drive(mount="/content/drive", retries=3):
    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            drive.mount(mount, force_remount=attempt > 1)
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            probe, tok = os.path.join(DRIVE_CACHE, ".mount_probe"), str(time.time())
            with open(probe, "w") as fh: fh.write(tok)
            with open(probe) as fh: got = fh.read()
            os.remove(probe)
            if got != tok: raise IOError("probe read-back mismatch")
            print(f"Drive OK (attempt {attempt})")
            return
        except Exception as e:
            print(f"[drive] attempt {attempt}/{retries}: {e}"); time.sleep(5 * attempt)
    raise RuntimeError("Drive would not mount.")

init_drive()
REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_frozen", "cache_finetune", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"; os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}"); sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe"
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe"), f"results/{c} does not resolve to Drive"
    os.remove(probe)

try:
    import torch
    gpu = torch.cuda.is_available()
except Exception:
    gpu = False
print(f"\nrepo: {os.getcwd()}")
print(f"GPU present: {gpu}  (not needed -- every backbone should be a cache hit)")
print(f"vCPUs: {os.cpu_count()}")

## 2. Datasets\n\nPaths only -- needed because `cache_key` hashes them, so they must match what produced the caches. The Drive-cached CelebA zip is used first, so no kaggle.json.

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
assert CELEBA_OK, "CelebA unavailable -- the Drive-cached zip should make this credential-free"
print("datasets ready |", os.environ["WATERBIRDS_ROOT"], "|", CELEBA_ROOT)

## 3. Gate: validator + what is actually on Drive

In [ ]:
import glob

rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_grid"],
                    capture_output=True, text=True)
print(rc.stdout[-1500:])
assert rc.returncode == 0, "grid validator FAILED"

print()
for c in ("cache_clip", "cache_resnet", "cache_frozen"):
    n = len(glob.glob(f"results/{c}/*"))
    sz = sum(os.path.getsize(p) for p in glob.glob(f"results/{c}/*") if os.path.isfile(p)) / 1e9
    print(f"  {c:14s} {n:4d} file(s), {sz:5.2f} GB")

## 4. Cache probe

Only the two cheapest cells are built here, so a missing cache surfaces in seconds rather than
after an hour. Each build aborts past three minutes: on a CPU runtime that means features are being
*computed*, which is hours of silent work. The grid then loads the remaining cells one at a time.

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip":   {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cpu",
                       "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cpu", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"},
            "frozen": {"device": "cpu", "cache_dir": "results/cache_frozen",
                       "batch_size": 128, "num_workers": 4}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

SLOW_SECONDS = 180

def build_cell(bb, ds):
    """One (backbone, dataset) GridData. A cache hit is seconds of Drive read; slower than that
    means this CPU runtime started COMPUTING features, which is hours of silent work."""
    t = time.time()
    gd = build_griddata(ds, bb, cfg_for(ds), seed=0)
    el = time.time() - t
    print(f"[loaded] {bb:14s}/{ds:10s} d={gd.train[0].shape[1]:5d} "
          f"train={gd.train[0].shape[0]:6d} eval={gd.eval_domain[0].shape[0]:6d} ({el:.0f}s)",
          flush=True)
    assert el < SLOW_SECONDS, (f"{bb}/{ds} took {el:.0f}s -- COMPUTED, not loaded. That cache is "
                               f"missing; re-extract it on a GPU runtime first.")
    return gd

import gc
for ds in DATASETS:                       # probe the cheapest backbone on both datasets
    gd = build_cell(BACKBONES[1], ds); del gd; gc.collect()
print()
print("cache verified on the probe cells; the grid loads the rest one at a time")

## 5. The grid (~5.5 h, streamed and resumable)

4 backbones x 2 datasets x 5 methods x 5 seeds x 3 scores x 6 rho x 10 splits = 36,000 records.

About 0.6 h for the four Waterbirds cells and 4.9 h for CelebA, whose eval pool is 5.7x larger and
whose training matrix is 34x taller. Each cell is written to Drive as it finishes, so **re-running
this cell after a disconnect resumes** rather than restarting.

Gated-out arms are evaluated and kept, tagged `gate_status`. The verdicts still exclude them, so
they keep their original meaning; `verdicts_with_excluded` is the sensitivity analysis itself.

In [ ]:
from study_robust_train.grid import run_grid_streaming

CELL_CSV = "results/study/grid_records.csv"
keys = [(bb, ds) for ds in DATASETS for bb in BACKBONES]   # Waterbirds first: cheap cells early

t = time.time()
out = run_grid_streaming(keys, build_cell, methods=METHODS, scores=SCORES,
                         rho_sweep=RHO_SWEEP, seeds=SEEDS, n_splits=N_SPLITS,
                         alpha=ALPHA, cell_csv=CELL_CSV)
print(f"\nelapsed {(time.time()-t)/60:.0f} min")
print(f"records  : {len(out['records']):,}")
print(f"excluded : {len(out['excluded'])} arm(s) below the worst-group floor")
print(f"flagged  : {len(out['flagged'])} arm(s) in the soft band")
print(f"failed   : {out['failed']}")
print(f"sensitivity verdicts available: {'verdicts_with_excluded' in out}")

by = {}
for r in out["records"]: by[r["gate_status"]] = by.get(r["gate_status"], 0) + 1
print("gate_status:", by)

done = sorted({(r["backbone"], r["dataset"]) for r in out["records"]})
print()
print(f"cells complete: {len(done)}/{len(keys)}")
if len(done) < len(keys):
    print("Re-run THIS cell to continue -- finished cells are read back from the CSV and skipped.")
else:
    print(f"persisted -> {CELL_CSV}")

## 6. R2.4: which arms were gated out, and what they would change

In [ ]:
# R2.4 asked for the influence of the excluded arms to be shown, not just their exclusion logged.
# They are now evaluated and kept, tagged gate_status="excluded", so the same verdicts can be
# recomputed with them included and the two compared.
if out["excluded"]:
    print("excluded arms (worst-group accuracy below the per-method floor):")
    for e in out["excluded"]:
        print(f"  {e['backbone']:14s}/{e['dataset']:10s} {e['method']:18s} seed {e['seed']} "
              f"wg={e['worst_group_acc']:.3f} < floor {e['floor']:.2f}")
    print()
    print("Compare the H1/H2/H3 verdicts with and without them -- out['verdicts'] excludes,")
    print("out['verdicts_with_excluded'] includes. Any conclusion that flips is a finding to")
    print("report, not a reason to hide the arm.")
else:
    print("no arm fell below its floor in this run")

if out["flagged"]:
    print("\nsoft-flagged (kept):")
    for f in out["flagged"]:
        print(f"  {f['backbone']:14s}/{f['dataset']:10s} {f['method']:18s} seed {f['seed']} "
              f"wg={f['worst_group_acc']:.3f} < expected {f['expected_min']:.2f}")

## 7. Recover the 50k CelebA ablation (free -- cache hits only)

In [ ]:
# Pure cache hits: the 50k features are still on Drive. This recovers the RECORDS for the
# abandoned subsample protocol so the comparison survives after those 7.5 GB are deleted.
if RUN_50K_ABLATION:
    from study_robust_train.representation import (build_repr_griddata,
                                                   run_representation_streaming,
                                                   write_representation_csv)
    def cfg50():
        return {"dataset": {"root": os.environ["CELEBA_ROOT"], "n_classes": 2},
                "finetune": {"device": "cpu", "batch_size": 128, "num_workers": 4,
                             "amp": True, "max_train": 50000, "cache_dtype": "float16",
                             "cache_dir": "results/cache_finetune", "select_by": ""}}
    HP = {"erm": dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
          "reweight": dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
          "groupdro": dict(optimizer="sgd", lr=1e-3, weight_decay=1e-2, groupdro_eta=0.05)}

    def build50(ds, obj, seed):
        t0 = time.time()
        gd = build_repr_griddata(ds, obj, cfg50(), ft_seed=seed,
                                 epochs=CEL50_EPOCHS[obj], **HP[obj])
        el = time.time() - t0
        assert el < 180, f"{obj}/s{seed} was COMPUTED ({el:.0f}s), not loaded -- 50k cache missing"
        print(f"[loaded 50k] celeba/{obj}/s{seed}  ({el:.0f}s)", flush=True)
        return gd

    keys50 = [("celeba", o, s) for o in ("erm", "groupdro", "reweight") for s in CEL50_SEEDS]
    o50 = run_representation_streaming(keys50, build50, heads=("erm", "dfr", "groupdro_ll"),
                                       scores=SCORES, n_splits=N_SPLITS, verbose=False)
    write_representation_csv(o50["records"], "results/study/representation_records_celeba50k.csv")
    print(f"\n50k records: {len(o50['records'])} | failed: {o50['failed']}")
    for sc in SCORES:
        lev = o50["verdicts"][f"celeba/{sc}"]["levers"]
        print(f"  {sc:5s} primary {lev['direction']:24s} | "
              + ", ".join(f"{h}={hv['direction']}" for h, hv in sorted(lev["by_head"].items())))
    print("\nThe cells that read marginal_wins here are indistinguishable at full train --")
    print("the counterexample was an artefact of the reduced minority count. Those 7.5 GB of")
    print("50k features are now safe to delete; this CSV keeps the evidence.")
else:
    print("skipped")

## 8. What is on Drive now

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/", check=False)
for f in ("grid_records.csv", "representation_records.csv",
          "representation_records_celeba50k.csv", "calibration_ablation.csv",
          "predicted_group_mondrian.csv"):
    p = f"{DRIVE_CACHE}/study/{f}"
    print(("FOUND   " if os.path.exists(p) else "MISSING ")
          + f + (f"  ({os.path.getsize(p)/1e6:.1f} MB)" if os.path.exists(p) else ""))

## 9. STOP -- what still needs writing

This run produces the records. The re-analysis that turns them into reviewer answers -- the
hierarchical cluster bootstrap on the main tables (R2.3), the equivalence test on the original
Table 2 (R3.1), and correlation CIs with the small-n caveat (R3.2) -- needs wiring that does not
exist yet: `stats.py` has the estimators, but they are not connected to the grid verdicts.

That is pure analysis over `grid_records.csv` and `calibration_ablation.csv`, so it costs no
compute and can be done locally once this CSV is on Drive.